<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">La Inferencia Optimizada</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Cuando un modelo ya está entrenado, el siguiente desafío es usarlo de forma eficiente en producción o en escenarios de uso intensivo. Este notebook introduce la inferencia optimizada como problema práctico distinto del entrenamiento.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Un modelo útil no solo debe acertar; también debe responder con costes razonables de tiempo y recursos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué significa optimizar inferencia y por qué esa etapa merece atención propia.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Entrenar bien y servir bien un modelo son problemas relacionados, pero no idénticos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Optimizar inferencia implica reducir latencia, consumo o complejidad sin degradar en exceso la calidad del resultado.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. Su cabeza de clasificación se inicializa de nuevo y no se entrena en este ejemplo.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes fundamentales para ejecutar inferencia optimizada en tareas de clasificación de secuencias.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados para inferencia optimizada.


In [ ]:
# Bloque 2 — Preparación de una entrada para inferencia
"""Este bloque define una entrada de texto para ejecutar inferencia.
Definimos un texto de entrada para el modelo.
La variable "texto_entrada" contiene un texto para evaluar el modelo en inferencia.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "entrada_tokenizada" mediante la expresión "tokenizador(texto_entrada, return_tensors="pt")".
El argumento "texto_entrada" indica el texto que se desea tokenizar.
El parámetro "return_tensors=" indica el tipo de tensor que se desea obtener, en este caso, "pt" para PyTorch.
La variable "entrada_tokenizada" almacena el texto de entrada tokenizado en formato tensor de PyTorch."""

texto_entrada = "El servicio fue excelente."

entrada_tokenizada = tokenizador(texto_entrada, return_tensors="pt")

entrada_tokenizada # Muestra el texto de entrada tokenizado en formato tensor.


In [ ]:
# Bloque 3 — Inferencia estándar
"""Este bloque ejecuta inferencia estándar sin desactivar el cálculo de gradientes.
Asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_estandar".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_estandar" contiene las salidas del modelo antes de realizar la operación argmax."""

logits_estandar = modelo(**entrada_tokenizada).logits

logits_estandar # Muestra los logits generados por el modelo antes de aplicar argmax.


In [ ]:
# Bloque 4 — Activación del modo no_grad para optimización
"""Este bloque activa el modo no_grad para reducir el uso de memoria y, según el caso, acelerar la inferencia.
Este bloque importa "torch" para trabajar con tensores y habilitar el modo no_grad durante la inferencia.

Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.
Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_no_grad".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo en el entorno no_grad.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_no_grad" contiene las salidas del modelo sin cálculo de gradientes."""

import torch

with torch.no_grad():
    logits_no_grad = modelo(**entrada_tokenizada).logits

logits_no_grad # Muestra los logits generados por el modelo sin cálculo de gradientes.


In [ ]:
# Bloque 5 — Cuantización dinámica del modelo
"""Este bloque aplica cuantización dinámica al modelo con el objetivo de reducir su tamaño.
Este bloque importa "copy" para crear una copia del modelo en memoria antes de aplicar la cuantización dinámica.
Este bloque importa "Int8DynamicActivationInt8WeightConfig" y "quantize_" de "torchao.quantization" para aplicar la cuantización dinámica al modelo.

Aplicamos la función "copy.deepcopy()" al modelo para crear una copia en memoria antes de aplicar la cuantización dinámica.
La variable "modelo_cuantizado" almacena la copia del modelo que será optimizada mediante cuantización dinámica.

Aplicamos la función "quantize_()" al modelo copiado y a la clase "Int8DynamicActivationInt8WeightConfig()" para realizar la cuantización dinámica.
Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original; su efecto en el rendimiento depende del entorno.
La variable "modelo_cuantizado" almacena ahora la versión cuantizada del modelo copiado, lista para inferencia."""

import copy
from torchao.quantization import (Int8DynamicActivationInt8WeightConfig, quantize_,)

modelo_cuantizado = copy.deepcopy(modelo)
quantize_(modelo_cuantizado, Int8DynamicActivationInt8WeightConfig())

# Esta operación cuantiza las capas compatibles del modelo copiado sin modificar el modelo original.


In [ ]:
# Bloque 6 — Inferencia con modelo cuantizado
"""Este bloque ejecuta inferencia utilizando el modelo cuantizado.
Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.

Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo_cuantizado()" a la variable "logits_cuantizado".
El argumento "**entrada_tokenizada" indica los tensores tokenizados que se pasan al modelo cuantizado durante la inferencia.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "entrada_tokenizada" y pasar sus elementos como argumentos al modelo.
La variable "logits_cuantizado" contiene las salidas del modelo cuantizado durante la inferencia."""

with torch.no_grad():
    logits_cuantizado = modelo_cuantizado(**entrada_tokenizada).logits

logits_cuantizado # Muestra las salidas del modelo cuantizado durante la inferencia y con los gradientes desactivados.


In [ ]:
# Bloque 7 — Batching para inferencia optimizada
"""Este bloque procesa múltiples entradas en un mismo batch; no mide ni demuestra una reducción de latencia.
Definimos un batch de textos de entrada para el modelo.
La variable "textos_batch" contiene una lista de textos para evaluar el modelo en inferencia.

Asignamos la llamada a la instancia de la clase "tokenizador()" a la variable "batch_tokenizado" mediante la expresión 
"tokenizador(textos_batch, padding=True, truncation=True, return_tensors="pt")".

El argumento "textos_batch" indica los textos que se desean tokenizar.
El parámetro "padding=" asegura que todas las secuencias del batch tengan la misma longitud mediante el relleno.
El parámetro "truncation=" recorta las secuencias que exceden la longitud máxima permitida por el modelo.
El parámetro "return_tensors=" indica el tipo de tensor que se desea obtener, en este caso, "pt" para PyTorch.
La variable "batch_tokenizado" almacena el batch tokenizado en tensores de PyTorch.

Aplicamos la clase "torch.no_grad()" mediante un bloque "with" para crear un entorno de gradientes desactivados durante la inferencia.
Dentro del entorno no_grad, asignamos el atributo ".logits" de la salida de "modelo()" a la variable "logits_batch".
El argumento "**batch_tokenizado" indica los tensores tokenizados que se pasan al modelo en el entorno no_grad.
Los asteriscos dobles "**" se utilizan para desempaquetar el diccionario "batch_tokenizado" y pasar sus elementos como argumentos al modelo.
La variable "logits_batch" contiene las salidas del modelo para todas las entradas del batch tokenizado."""

textos_batch = ["El producto es excelente.", "La experiencia fue terrible.", "El envío llegó rápido."]

batch_tokenizado = tokenizador(textos_batch, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    logits_batch = modelo(**batch_tokenizado).logits


logits_batch # Muestra los logits resultantes para el batch tokenizado.


In [ ]:
# Bloque 8 — Resultados finales
"""Este bloque organiza los resultados del proceso de inferencia optimizada.
La variable "resultado" almacena en forma de diccionario las salidas estándar, optimizadas y cuantizadas."""

resultado = {
    "logits_estandar": logits_estandar,
    "logits_no_grad": logits_no_grad,
    "logits_cuantizado": logits_cuantizado,
    "logits_batch": logits_batch
}

resultado # Muestra el diccionario con los resultados finales.

<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto la inferencia optimizada como una etapa práctica orientada a hacer viable el uso real del modelo.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Rendimiento y calidad deben equilibrarse según el contexto de uso del sistema.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Asumir que un modelo entrenado con buen desempeño está automáticamente listo para desplegarse eficientemente.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Exportación de los Modelos para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Exportación de los Modelos y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">08_Inferencia_optimizada/02_La_exportacion_de_los_modelos.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">